# Diagnosing a climate model with a Gregory plot

**Week 5 · From the linear EBM to CMIP6**

The forcing-feedback relation is:
$$N'=F+\alpha T',\qquad\alpha<0\text{ for a restoring response}.$$

Can this describe a much more complex climate model?

By the end of this activity, you should be able to:

- Explain the control and abrupt CO₂ experiments used to construct a Gregory plot.
- Calculate global temperature and top-of-atmosphere radiation anomalies.
- Interpret the fitted forcing, net feedback parameter, and effective climate sensitivity.

The data access, averaging, and plotting code are supplied. Read each prediction prompt before running the following cells, and write your interpretations in the response cells. No new numerical integration is needed. Run cells in order.

We use actual **MPI-ESM1-2-LR** CMIP6 output, one realization (`r1i1p1f1`), on its native atmospheric grid (`gn`). This diagnoses one model; it is not an observational estimate of Earth's sensitivity.

## 1. What experiment are we analyzing?

In **`piControl`**, the external conditions are held at preindustrial values, while weather and internal climate variability continue. **`abrupt-4xCO2`** branches from that control: CO₂ concentration is abruptly quadrupled and then held fixed. We analyze the first 150 years after the increase, together with the corresponding control segment.

Abrupt quadrupling is a standard CMIP6 core experiment and is widely used for Gregory estimates ([experiment definitions](https://wcrp-cmip.github.io/CMIP6_CVs/docs/CMIP6_experiment_id.html)). It is an idealized forcing-step experiment, not a future emissions scenario. The model does not need to have reached equilibrium by year 150.

**Predict before running:** After CO₂ increases, what do you expect to happen to global temperature and the net incoming radiation? Why can warming continue while CO₂ is constant?

**Your response:**

### Setup

Run the cells in order in the course Python environment. The data preparation and plotting code are supplied.

In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
import hashlib
import importlib.metadata
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import cftime
from IPython.display import display

get_ipython().run_line_magic('matplotlib', 'inline')

plt.rcParams.update({'figure.figsize': (8, 4.5), 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})

MODEL = 'MPI-ESM1-2-LR'
MEMBER = 'r1i1p1f1'
GRID = 'gn'
VERSION = 'v20190710'
NYEARS = 150
VARIABLES = ['tas', 'rsdt', 'rsut', 'rlut']
EXPERIMENTS = ['piControl', 'abrupt-4xCO2']
CATALOG_URL = 'https://storage.googleapis.com/cmip6/pangeo-cmip6.csv'

# The default is a fresh calculation from cloud-hosted spatial fields.
# True reuses the small annual-mean files made by a previous successful run.
# Keep both the CSV and its provenance JSON together in the data folder.
USE_SAVED_MEANS = False
DATA_DIR = Path('data')
MEANS_FILE = DATA_DIR / 'annual_global_means.csv'
PROVENANCE_FILE = DATA_DIR / 'annual_global_means.provenance.json'
settings = dict(model=MODEL, member=MEMBER, grid=GRID, version=VERSION,
                years=NYEARS, variables=VARIABLES, processing='cell-area_month-days_v1')
run_start = perf_counter()

## 2. Prepare the annual global means

Run the supplied cells to obtain annual global temperature and radiation for each experiment.

| Variable | Meaning | Units |
|---|---|---|
| `tas` | Near-surface air temperature | K |
| `rsdt` | Incoming sunlight at the top of the atmosphere | W m⁻² |
| `rsut` | Reflected sunlight | W m⁻² |
| `rlut` | Outgoing longwave radiation | W m⁻² |

The global-mean net radiation is $N=rsdt-rsut-rlut$. **Positive $N$ means energy entering the climate system.**


In [ ]:
if USE_SAVED_MEANS:
    provenance = json.loads(PROVENANCE_FILE.read_text())
    if provenance['settings'] != settings:
        raise ValueError('Saved means use different settings. Run with USE_SAVED_MEANS=False.')
    if hashlib.sha256(MEANS_FILE.read_bytes()).hexdigest() != provenance['csv_sha256']:
        raise ValueError('The saved means and provenance do not match.')
    annual = pd.read_csv(MEANS_FILE)
    print('Using previously calculated annual means; no cloud fields read this run.')
else:
    catalogue = pd.read_csv(CATALOG_URL, low_memory=False)
    model_rows = catalogue.loc[
        (catalogue.source_id == MODEL) & (catalogue.member_id == MEMBER)
        & (catalogue.grid_label == GRID)]

    def locate(experiment, variable, table='Amon'):
        rows = model_rows.loc[
            (model_rows.experiment_id == experiment)
            & (model_rows.variable_id == variable)
            & (model_rows.table_id == table)
            & model_rows.zstore.str.rstrip('/').str.endswith('/' + VERSION)]
        stores = rows.zstore.drop_duplicates().tolist()
        if len(stores) != 1:
            raise ValueError(f'Expected one store for {experiment}/{variable}; found {len(stores)}.')
        return stores[0]

    sources = {exp: {var: locate(exp, var) for var in VARIABLES}
               for exp in EXPERIMENTS}
    area_source = locate('piControl', 'areacella', 'fx')
    display(pd.DataFrame([{'experiment': exp, 'variable': var, 'store': url}
                          for exp in EXPERIMENTS for var, url in sources[exp].items()]))

In [ ]:
if not USE_SAVED_MEANS:
    # Zarr opens metadata first; field data are read when their means are computed.
    def open_cloud(url):
        return xr.open_zarr(url, consolidated=True, chunks={},
                            decode_times=xr.coders.CFDatetimeCoder(use_cftime=True),
                            storage_options={'token': 'anon'})

    datasets = {exp: {var: open_cloud(url) for var, url in sources[exp].items()}
                for exp in EXPERIMENTS}
    perturbed = datasets['abrupt-4xCO2']['tas']
    control = datasets['piControl']['tas']
    attrs = perturbed.attrs
    assert attrs['parent_experiment_id'] == 'piControl'
    assert attrs['parent_variant_label'] == MEMBER
    calendar = control.time.encoding.get('calendar', 'standard')
    branch = cftime.num2date(attrs['branch_time_in_parent'],
                            attrs['parent_time_units'], calendar=calendar)
    if (branch.month, branch.day) != (1, 1):
        raise ValueError('This teaching notebook expects a January 1 control branch.')
    first_control_year = branch.year

    # Select the matched control segment BEFORE reading the spatial fields.
    selected = {}
    selection_dates = {}
    for exp in EXPERIMENTS:
        selected[exp] = {}
        for var, ds in datasets[exp].items():
            assert ds.attrs['source_id'] == MODEL
            assert ds.attrs['variant_label'] == MEMBER
            assert ds.attrs['grid_label'] == GRID
            if exp == 'piControl':
                field = ds[var].sel(time=slice(f'{first_control_year}-01-01',
                                              f'{first_control_year + NYEARS - 1}-12-31'))
            else:
                assert ds.attrs['branch_time_in_parent'] == attrs['branch_time_in_parent']
                assert ds.attrs['parent_time_units'] == attrs['parent_time_units']
                field = ds[var].isel(time=slice(0, 12 * NYEARS))
            # Require 150 complete consecutive January–December years.
            assert field.sizes['time'] == 12 * NYEARS
            months = field.time.dt.month.values
            years = field.time.dt.year.values
            np.testing.assert_array_equal(months, np.tile(np.arange(1, 13), NYEARS))
            np.testing.assert_array_equal(years, np.repeat(np.arange(years[0], years[0]+NYEARS), 12))
            selected[exp][var] = field
        xr.align(*selected[exp].values(), join='exact')
        dates = selected[exp]['tas'].time
        selection_dates[exp] = [str(dates.values[0]), str(dates.values[-1])]

    area = open_cloud(area_source).areacella.astype('float64').compute()
    assert bool(np.isfinite(area).all()) and bool((area > 0).all())
    # The area file's latitude labels differ by ~1e-14 degrees in this archive.
    # Verify identical cells to tight tolerance before normalizing their labels.
    reference_grid = selected['piControl']['tas']
    for coordinate in ['lat', 'lon']:
        np.testing.assert_allclose(area[coordinate], reference_grid[coordinate],
                                   rtol=0, atol=1e-10)
    area = area.assign_coords(lat=reference_grid.lat, lon=reference_grid.lon)
    for exp in EXPERIMENTS:
        for field in selected[exp].values():
            xr.align(field.isel(time=0, drop=True), area, join='exact')
    earth_area = 4 * np.pi * 6_371_000**2
    assert np.isclose(float(area.sum()) / earth_area, 1, rtol=0.02)
    print(f'Control branch: {branch}; calendar: {calendar}')
    display(pd.DataFrame(selection_dates, index=['first monthly timestamp', 'last monthly timestamp']).T)
    print(f'Grid: {area.sizes["lat"]} × {area.sizes["lon"]}; area sum: {float(area.sum()):.3e} m²')
else:
    display(pd.DataFrame(provenance['selection_dates'], index=['first monthly timestamp', 'last monthly timestamp']).T)

In [ ]:
if not USE_SAVED_MEANS:
    averaging_start = perf_counter()
    tables, timings = [], []
    for exp in EXPERIMENTS:
        table = pd.DataFrame({'experiment': exp, 'year': np.arange(1, NYEARS + 1)})
        for var in VARIABLES:
            start = perf_counter()
            field = selected[exp][var]
            # Weight cells by model-provided area, then months by their calendar length.
            monthly_mean = field.weighted(area).mean(('lat', 'lon'), skipna=False)
            days = field.time.dt.days_in_month.astype('float64')
            annual_mean = ((monthly_mean * days).groupby('time.year').sum(skipna=False)
                           / days.groupby('time.year').sum())
            # Two local Dask threads process one field at a time; no cluster is needed.
            values = annual_mean.compute(scheduler='threads', num_workers=2).values
            assert len(values) == NYEARS and np.isfinite(values).all()
            table[var] = values
            elapsed = perf_counter() - start
            timings.append(dict(experiment=exp, variable=var, seconds=elapsed))
            print(f'{exp:14s} {var:4s}: {elapsed:.1f} s', flush=True)
        table['N'] = table.rsdt - table.rsut - table.rlut
        tables.append(table)
    annual = pd.concat(tables, ignore_index=True)
    averaging_seconds = perf_counter() - averaging_start
    DATA_DIR.mkdir(exist_ok=True)
    annual.to_csv(MEANS_FILE, index=False, float_format='%.12g')
    packages = ['numpy', 'pandas', 'matplotlib', 'xarray', 'dask', 'zarr', 'gcsfs', 'cftime']
    provenance = dict(
        settings=settings, created_utc=datetime.now(timezone.utc).isoformat(),
        catalogue=CATALOG_URL, sources=sources, area_source=area_source,
        selection_dates=selection_dates, parent_branch=str(branch), calendar=calendar,
        area_sum_m2=float(area.sum()), averaging_seconds=averaging_seconds,
        setup_and_averaging_seconds=perf_counter()-run_start, field_timings=timings,
        packages={p: importlib.metadata.version(p) for p in packages},
        license=attrs.get('license', ''), further_info_url=attrs.get('further_info_url', ''),
        csv_sha256=hashlib.sha256(MEANS_FILE.read_bytes()).hexdigest())
    PROVENANCE_FILE.write_text(json.dumps(provenance, indent=2))
    print(f'Cloud-field averaging: {averaging_seconds:.1f} s ({averaging_seconds/60:.2f} min).')

# These checks also run when reusing saved means.
assert set(annual.experiment) == set(EXPERIMENTS)
assert not annual.duplicated(['experiment', 'year']).any()
assert np.isfinite(annual[VARIABLES + ['N']].to_numpy()).all()
for exp in EXPERIMENTS:
    np.testing.assert_array_equal(annual.loc[annual.experiment == exp, 'year'], np.arange(1, NYEARS+1))
np.testing.assert_allclose(annual.N, annual.rsdt-annual.rsut-annual.rlut, atol=1e-8)
display(annual.groupby('experiment')[['tas', 'N']].mean().rename(
    columns={'tas': 'mean temperature (K)', 'N': 'mean net radiation (W m-2)'}))

## 3. Changes relative to which climate?

Our equation is about anomalies. We need both
$$T'(t)=T_{4\times}(t)-\widehat T_c(t),\qquad
N'(t)=N_{4\times}(t)-\widehat N_c(t).$$

The hats denote estimates of the control's background climate. A control may have a small radiation offset and a slow drift. We first inspect its annual series. For this activity, we fit a linear background trend to the matched 150 control years, separately for temperature and radiation, and subtract that background from the perturbed run. This removes estimated linear drift without subtracting the control's individual weather fluctuations. A fitted trend is an approximation, not proof that every slow change is model drift.

The baseline comes from the model’s control simulation.

In [ ]:
control_annual = annual.loc[annual.experiment == 'piControl'].set_index('year')
forced_annual = annual.loc[annual.experiment == 'abrupt-4xCO2'].set_index('year')
year = control_annual.index.to_numpy()
control_background = {}
control_trends = {}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
for ax, variable, label in zip(axes, ['tas', 'N'],
                             ['Control temperature (K)', 'Control net radiation (W m$^{-2}$)']):
    coefficients = np.polyfit(year, control_annual[variable], 1)
    control_background[variable] = np.polyval(coefficients, year)
    control_trends[variable] = coefficients[0] * 100
    ax.plot(year, control_annual[variable], color='0.65', lw=1, label='Annual control')
    ax.plot(year, control_background[variable], color='#326b9b', lw=2, label='Linear background')
    ax.set(xlabel='Year relative to branch', ylabel=label)
    ax.legend()
plt.show()
print(f'Fitted control temperature trend: {control_trends["tas"]:+.4f} K per century')
print(f'Fitted control radiation trend: {control_trends["N"]:+.4f} W m⁻² per century')

T_prime = forced_annual.tas.to_numpy() - control_background['tas']
N_prime = forced_annual.N.to_numpy() - control_background['N']

**Interpret:** Why subtract the control radiation as well as the control temperature? Why do we subtract a smooth background rather than individual control years?

**Your response:**

## 4. Watch the response unfold in time

These are two views of the same forced experiment. Temperature is the accumulated response; net radiation is an energy input rate. A declining imbalance can remain positive while temperature keeps increasing.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
axes[0].plot(year, T_prime, color='#b54b32', lw=1.5)
axes[1].plot(year, N_prime, color='#326b9b', lw=1.5)
for ax in axes:
    ax.axhline(0, color='0.5', lw=0.8)
    ax.set_xlabel('Year after CO₂ increase')
axes[0].set_ylabel("Temperature anomaly T′ (K)")
axes[1].set_ylabel("Net radiation anomaly N′ (W m$^{-2}$)")
fig.suptitle(f'{MODEL}: abrupt-4×CO₂ relative to control background')
plt.show()
print(f'Years 141–150 mean warming: {T_prime[-10:].mean():.2f} K')
print(f'Years 141–150 mean imbalance: {N_prime[-10:].mean():.2f} W m⁻²')

**Interpret:** Has the climate reached equilibrium by the end of the displayed period? Which quantity most directly supports your answer? Does a slower warming rate mean that the planet has stopped gaining energy?

**Your response:**

## 5. Put radiation against warming: the Gregory plot

Put $T'$ on the horizontal axis and $N'$ on the vertical axis. Each point is one year. Color records elapsed time, which otherwise disappears from the axes.

**Predict:** Which direction should the points generally move as time passes? In $N'=F+\alpha T'$, what are the vertical intercept, slope, and horizontal intercept? What units does each have?

**Your response:**

In [ ]:
def gregory_fit(temperature, imbalance):
    """Ordinary least squares: N′ = intercept + slope*T′."""
    slope, intercept = np.polyfit(temperature, imbalance, 1)
    if slope >= 0:
        raise ValueError('The fitted slope is not restoring; a positive equilibrium estimate is inappropriate.')
    return dict(forcing=intercept, slope=slope, alpha=slope,
                warming_4x=-intercept/slope, ecs_eff=-intercept/(2*slope))

fit = gregory_fit(T_prime, N_prime)
fig, ax = plt.subplots(figsize=(8, 5), layout='constrained')
points = ax.scatter(T_prime, N_prime, c=year, cmap='viridis', s=26, alpha=0.85)
fig.colorbar(points, ax=ax, label='Year after CO₂ increase')
x_observed = np.linspace(T_prime.min(), T_prime.max(), 100)
x_extended = np.linspace(0, max(fit['warming_4x'], T_prime.max()) * 1.08, 150)
ax.plot(x_extended, fit['forcing'] + fit['slope']*x_extended,
        '--', color='#b54b32', lw=1.3, label='Line extended beyond observations')
ax.plot(x_observed, fit['forcing'] + fit['slope']*x_observed,
        color='#b54b32', lw=2.2, label='OLS fit, years 1–150')
ax.scatter([0, fit['warming_4x']], [fit['forcing'], 0],
           marker='x', s=65, color='#b54b32', zorder=5)
ax.axhline(0, color='0.5', lw=0.8)
ax.axvline(0, color='0.5', lw=0.8)
ax.set(xlabel="Temperature anomaly T′ (K)", ylabel="Net radiation anomaly N′ (W m$^{-2}$)",
       title=f'{MODEL}: Gregory plot')
ax.legend(loc='upper right', fontsize=9)
plt.show()

display(pd.DataFrame({
    'Quantity': ['Fitted quadrupling forcing', 'Slope dN′/dT′', 'Net feedback parameter α',
                 'Extrapolated quadrupling warming', 'Effective doubled-CO₂ sensitivity'],
    'Value': [fit['forcing'], fit['slope'], fit['alpha'], fit['warming_4x'], fit['ecs_eff']],
    'Units': ['W m⁻²', 'W m⁻² K⁻¹', 'W m⁻² K⁻¹', 'K', 'K']}).round(3))

### Translate the line back into climate quantities

For a fitted line $N'=a+bT'$,
$$\widehat F_{4\times}=a,\qquad\widehat\alpha=b,
\qquad\widehat T'_{\mathrm{eq},4\times}=-\frac{a}{b}.$$

We use **negative $\alpha$ for a restoring response**. The intercept estimates an effective forcing, including rapid adjustments; it is not simply the first year's measured imbalance. The horizontal intercept extrapolates the warming at zero anomalous net radiation.

ECS refers to **doubled**, not quadrupled, CO₂. We use one additional fact from radiative-transfer calculations: **CO₂ forcing is approximately logarithmic in concentration, so each doubling adds approximately the same forcing.** Quadrupling is two doublings, giving $F_{4\times}\approx2F_{2\times}$.

Assuming the same net feedback parameter $\alpha$ for the two experiments, equilibrium warming $-F/\alpha$ also doubles. We therefore halve the **extrapolated temperature intercept** to estimate effective ECS:
$$\widehat{\mathrm{ECS}}_{\mathrm{eff}}\approx\frac{\widehat T'_{\mathrm{eq},4\times}}{2}=-\frac{a}{2b}.$$

We take the logarithmic forcing relationship as a given here; deriving its absorption physics is outside this activity.

This is an **effective sensitivity from a finite-period fit**. It need not equal the warming the model would reach after a much longer simulation. Evolving warming patterns and climate-state-dependent feedbacks can bend the relationship. Regression scatter is not the only source of uncertainty, so we do not label a simple line-fit confidence interval as uncertainty in the true ECS.

**Interpret:**

1. Report $F_{4\times}$, $\alpha$, and effective ECS with units. Is the horizontal intercept a doubling or quadrupling response?
2. Why does the fitted forcing differ from the first annual value of $N'$?
3. If $F$ stayed fixed and $\alpha$ became more negative, how would the line and its horizontal intercept change?

**Your response:**

## Sources and reproducibility

- Gregory et al. (2004), [A new method for diagnosing radiative forcing and climate sensitivity](https://doi.org/10.1029/2003GL018747): the diagnostic idea.
- Henry Drake and Ryan Abernathey, [Pangeo CMIP6 Gregory notebook](https://gallery.pangeo.io/repos/pangeo-gallery/cmip6/ECS_Gregory_method.html): the starting workflow. This teaching adaptation uses one model, explicit cell areas, calendar-weighted annual means, branch-aligned control data, and a documented baseline.
- [IPCC AR6 WGI Chapter 7, Box 7.1](https://www.ipcc.ch/report/ar6/wg1/chapter/chapter-7/): forcing, feedback, and effective sensitivity.
- Zehrung et al. (2025), [Standardising the “Gregory method”](https://gmd.copernicus.org/articles/18/9433/2025/): processing choices. Our linear control background is a transparent teaching choice; it is not their recommended rolling-mean implementation.

**Data acknowledgment:** CMIP6 MPI-ESM1-2-LR output produced by MPI-M, accessed through the Pangeo cloud archive. The annual series are derived from the model output and retain its CC BY-SA 4.0 attribution/share-alike terms. The saved provenance records exact stores, version, branch, calendar, processing, package versions, timings, and the source's further-information link. These intermediate means are sufficient to reproduce the regression without reading the spatial fields again.